In [7]:
import os
import sys
import pickle
import pandas as pd
from pathlib import Path
from src.rctgan import Metadata
from src.rctgan.rctgan import RCTGAN
import warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, '/home/yjung/SynPriv')

exp_name = "rctgan_retail"
sample_sizes = [10]  # 100, 500, 1000, 3000, 5000, 10000, 15000]
data_path = Path("../data/sampled")
base_save_dir = Path(f"../results/{exp_name}")
n_samples = 10  # 생성할 메타 테이블 샘플 수

def run_rctgan(sample_size):
    sub_exp_name = f"train{sample_size}"
    save_dir = base_save_dir / sub_exp_name
    save_dir.mkdir(parents=True, exist_ok=True)

    meta_path = data_path / f"retail_sample_{sample_size}_meta.csv"
    flow_path = data_path / f"retail_sample_{sample_size}_flow.csv"
    df_meta = pd.read_csv(meta_path)
    df_flow = pd.read_csv(flow_path)
    df_flow["flow_id"] = df_flow.index.astype(str)
    tables = {"meta": df_meta, "flow": df_flow}

    metadata = Metadata()
    meta_fields = {
        'fkey': {'type': 'id', 'subtype': 'string'},
        'date': {'type': 'datetime', 'format': '%Y-%m-%d %H:%M:%S'},
        'CustomerID': {'type': 'categorical'},
        'Country': {'type': 'categorical'}
    }
    flow_fields = {
        'flow_id': {'type': 'id', 'subtype': 'string'},
        'fkey': {'type': 'id', 'subtype': 'string'},
        'Description': {'type': 'categorical'},
        'Quantity': {'type': 'numerical', 'subtype': 'float'},
        'UnitPrice': {'type': 'numerical', 'subtype': 'float'}
    }

    metadata.add_table(name="meta", data=df_meta, primary_key="fkey", fields_metadata=meta_fields)
    metadata.add_table(name="flow", data=df_flow, primary_key="flow_id", fields_metadata=flow_fields)
    metadata.add_relationship(parent="meta", child="flow", foreign_key="fkey")

    with open(save_dir / "retail_metadata.pkl", "wb") as f:
        pickle.dump(metadata, f)

    custom_hyperparams = {"meta": {"epochs": 10}, "flow": {"epochs": 10}}
    model = RCTGAN(model_kwargs={"hyperparam": custom_hyperparams})
    model.fit(metadata=metadata, tables=tables)

    with open(save_dir / "retail_rctgan_model.pkl", "wb") as f:
        pickle.dump(model, f)

    sampled = model.sample(num_rows={"meta": n_samples})
    for table_name, df in sampled.items():
        df.to_csv(save_dir / f"syn_{table_name}_{n_samples}.csv", index=False)

    syn_df = sampled["meta"].merge(sampled["flow"], on="fkey", how="inner")
    syn_df.to_csv(save_dir / f"syn_rctgan_{n_samples}.csv", index=False)
    print(f"{sample_size} 완료: {save_dir}")

for size in sample_sizes:
    run_rctgan(size)

100%|██████████| 10/10 [00:00<00:00, 56.57it/s]


10 완료: ../results/rctgan_retail/train10
